# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. This consists of ordered logistic regression outputs on knowledge adoption in rangeland management practices in Northern Kenya.

### Dataset Source
The dataset is described by a Croissant schema, accessible at the following URL:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review record sets and available fields. All identifiers (`@id`) are displayed for reference.

In [ ]:
# List available record sets and their fields, referencing by @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets were found in the dataset. Please review the dataset metadata or download links.")
else:
    for rec_set in record_sets:
        print(f"\nRecord Set @id: {rec_set['@id']}")
        print(f"  Name: {rec_set.get('name', '')}")
        print(f"  Fields:")
        for field in rec_set.get('field', []):
            if isinstance(field, dict):
                print(f"    - {field.get('@id', field)} ({field.get('name', '')})")
            else:
                print(f"    - {field}")
        print("")

# If example record set exists, display a preview of a record
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print(f"Example from Record Set {example_record_set_id}:")
    try:
        example_record = next(dataset.records(record_set=example_record_set_id))
        print(example_record)
    except StopIteration:
        print("No records found in this record set.")

## 3. Data Extraction
Load all records from each record set into Pandas DataFrames. All references are by their `@id`s.

In [ ]:
# Prepare to extract data into DataFrames
record_set_ids = [rec_set['@id'] for rec_set in dataset.record_sets]
dataframes = {}

for rec_set_id in record_set_ids:
    records = list(dataset.records(record_set=rec_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_set_id] = df
        print(f"Loaded {len(df)} records for Record Set {rec_set_id}")
    else:
        print(f"Record Set {rec_set_id} contains no records.")

# Display columns of the first DataFrame if any
if dataframes:
    first_rec_set_id = next(iter(dataframes.keys()))
    print(f"\nFields (@id as column names) in record set {first_rec_set_id}:")
    print(dataframes[first_rec_set_id].columns.tolist())
    display(dataframes[first_rec_set_id].head())
else:
    print("No dataframes were created.")

## 4. Exploratory Data Analysis (EDA)
We demonstrate basic data processing steps such as filtering, normalization, and grouping for a selected numeric field, using `@id` references. Modify `numeric_field_id` and `group_field_id` as appropriate for your analysis.

In [ ]:
# Example EDA: Select a DataFrame and a numeric field for processing
import numpy as np

# Ensure at least one DataFrame with numeric fields exists
if dataframes:
    # Select the first loaded DataFrame for demonstration
    record_set_id = first_rec_set_id
    df = dataframes[record_set_id]
    print(f"Working with Record Set: {record_set_id}")
    # Try to auto-detect a numeric field (column) by dtype
    numeric_field_id = None
    for col in df.columns:
        try:
            # Try converting to numeric sample
            sample_val = pd.to_numeric(df[col].dropna().iloc[0])
            numeric_field_id = col
            break
        except:
            continue
    if numeric_field_id is None:
        print("No numeric field detected for this record set.")
    else:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Remove outliers/NaN
        filtered_df = df[df[numeric_field_id].notnull()]
        threshold = filtered_df[numeric_field_id].mean() # Use mean as example threshold
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Filtered records in {record_set_id} with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())
        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} values:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        # Attempt to group by a non-numeric, non-unique field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < len(df)//2:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric group field detected.")
else:
    print("No dataframes to analyze. Please check previous steps or the dataset structure.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its grouping (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(filtered_df[numeric_field_id], ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field_id}")

    if 'grouped_df' in locals():
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ax=axes[1])
        axes[1].set_title(f"Mean {numeric_field_id} by {group_field_id}")
        axes[1].tick_params(axis='x', rotation=45)
    else:
        axes[1].remove()
    plt.tight_layout()
    plt.show()
else:
    print("No filtered data available for visualization.")

## 6. Conclusion
In this notebook, we have demonstrated loading a Croissant-formatted dataset via `mlcroissant`, inspecting available record sets using their `@id`, extracting records as DataFrames, performing numeric filtering and normalization, and visualizing patterns. All entities are referenced only by their `@id` to ensure reproducibility and alignment with the dataset schema.Continue with deeper domain analysis as needed!